# Self-dual residual trees: a computational companion

This notebook accompanies **“Self-Dual Residual Trees by Synchronized Evolution of Component Trees.”** It assumes that the paper is open beside the notebook. Definitions and proofs stay in the paper; here we make the constructions observable, connect the notation to the public `mmcfilters` API, and check the main structural properties on the exact $17\times17$ example from Figure 1.

The notebook does **not** reimplement Algorithm 1. The library constructs each final valued tree. We then read its recorded supports and signed residues to recover the scale-space governed by Equation (5).

## Goal

By the end, you will be able to:

- recreate the six-flat-zone image used in Figure 1;
- construct the unrestricted and saturated residual trees with one shared 4-adjacency;
- recover and visualize $I^0,\ldots,I^q$ from the valued residual tree;
- inspect the supports, proper parts, valuations, and signed residuals;
- check Equation (6), Proposition 2, Proposition 3, and Theorem 2 computationally;
- see why saturation under a shared adjacency does not by itself produce a tree of shapes;
- compare area pruning and the effect of changing the shared adjacency;
- reproduce the Figure 2 hydrant experiment with its exact image, hierarchies, and thresholds.

## Setup

Run this notebook with the project environment described in the repository `README.md` and `notebooks/requirements.txt`. It uses the installed `mmcfilters` package together with NumPy, pandas, Matplotlib, and IPython from the notebook environment.

In [1]:
import mmcfilters
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from collections import deque
from IPython.display import display
from matplotlib.colors import BoundaryNorm, ListedColormap
from pathlib import Path

print(f"mmcfilters: {mmcfilters.__version__}")

mmcfilters: 4.0.0


### Key assumptions

- Images are two-dimensional, C-contiguous `np.uint8` arrays.
- `radius=1.0` selects the shared 4-adjacency used for both polarities in Figure 1.
- Pixel `0`, the upper-left pixel in row-major order, is $p_\infty$.
- The default tie policy is the contrast-invariant spatial policy from Equation (4).
- The event order can be recovered from the final tree by sorting non-root supports by $(|X|,\min_\prec X)$. The paper proves that selected cardinalities do not decrease, and a newly merged candidate strictly contains the selected support.
- The residual-tree factories used here have one shared symmetric adjacency. The complementary-adjacency specialization of Theorem 3 is a separate topographic construction and is not presented as a shared-adjacency API call.
- The later filtering comparison constructs its tree of shapes with the complementary Min4/Max8 convention used in Figure 2.

## Steps

### 1. Recreate the image from Figure 1

The manuscript defines six 4-connected flat zones: exterior $C$, bright ring $P$, intersection $R$, hole $H$, lower arm $Q$, and bright exterior zone $D$. Their gray levels are $0,2,1,0,0,2$, respectively. The code below translates the exact geometric predicates used to generate the figure.

In [2]:
ROWS = COLUMNS = 17
row, column = np.mgrid[1 : ROWS + 1, 1 : COLUMNS + 1]

in_base_circle = (row - 9) ** 2 + (column - 9) ** 2 <= 37
in_lip = ((row == 4) | (row == 14)) & (column == 13)
in_shape = in_base_circle | in_lip

in_intersection = in_shape & (
    (column >= 14) | ((column >= 13) & (row >= 5) & (row <= 13))
)
in_hole = (
    in_shape
    & ~in_intersection
    & (14 * (row - 9) ** 2 + 20 * (column - 8.25) ** 2 <= 280)
)
in_lower_arm = (
    ((row == 4) & (column == 14))
    | ((row == 5) & (column >= 14) & (column <= 16))
    | (((row == 6) | (row == 7)) & (column >= 15) & (column <= 16))
    | ((row >= 8) & (row <= 10) & (column == 16))
    | (((row == 11) | (row == 12)) & (column >= 15) & (column <= 16))
    | ((row == 13) & (column >= 14) & (column <= 16))
    | ((row == 14) & (column == 14))
)
in_high_exterior = (
    (column == 17)
    | (((row == 3) | (row == 15)) & (column >= 14) & (column <= 16))
    | (((row == 4) | (row == 14)) & (column >= 15) & (column <= 16))
)

zone_masks = {
    "P": in_shape & ~in_intersection & ~in_hole,
    "R": in_intersection,
    "H": in_hole,
    "Q": in_lower_arm,
    "D": in_high_exterior,
}
zone_masks["C"] = ~np.logical_or.reduce(list(zone_masks.values()))

zone_membership_count = np.sum(list(zone_masks.values()), axis=0)
assert np.all(zone_membership_count == 1), "Flat zones must partition the domain."

zone_altitudes = {"C": 0, "P": 2, "R": 1, "H": 0, "Q": 0, "D": 2}
image = np.zeros((ROWS, COLUMNS), dtype=np.uint8)
for zone_name, mask in zone_masks.items():
    image[mask] = zone_altitudes[zone_name]
image = np.ascontiguousarray(image)

gray_cmap = ListedColormap(["#111111", "#888888", "#eeeeee"])
gray_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], gray_cmap.N)

zone_summary = pd.DataFrame(
    [
        {
            "zone": name,
            "gray level": zone_altitudes[name],
            "cardinality": int(mask.sum()),
        }
        for name, mask in zone_masks.items()
    ]
).set_index("zone")

fig, ax = plt.subplots(figsize=(4.8, 4.5), constrained_layout=True)
view = ax.imshow(image, cmap=gray_cmap, norm=gray_norm, interpolation="nearest")
ax.set_title(r"Figure 1 input $I^0$")
ax.set_xlabel("column")
ax.set_ylabel("row")
ax.set_xticks(range(0, COLUMNS, 2), labels=range(1, COLUMNS + 1, 2))
ax.set_yticks(range(0, ROWS, 2), labels=range(1, ROWS + 1, 2))
fig.colorbar(view, ax=ax, ticks=[0, 1, 2], label="Gray level", shrink=0.78)
plt.show()

display(zone_summary)

/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42385/4241428736.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,gray level,cardinality
zone,,
P,2,51
R,1,19
H,0,53
Q,0,19
D,2,27
C,0,120


### 2. Build the two shared-adjacency residual trees

The two calls differ only in the saturated eligibility restriction and the exterior point. Both return a valued morphological tree with exact image reconstruction.

In [3]:
factory = mmcfilters.MorphologicalTreeFactory
ADJACENCY_RADIUS = 1.0
INFINITY_PIXEL = 0

trees = {
    "Unrestricted": factory.create_unrestricted_residual_tree(
        image,
        radius=ADJACENCY_RADIUS,
    ),
    "Saturated": factory.create_saturated_residual_tree(
        image,
        infinity_pixel=INFINITY_PIXEL,
        radius=ADJACENCY_RADIUS,
    ),
}


def enum_name(value) -> str:
    return str(value).split(".")[-1]


def adjacency_name(tree) -> str:
    context = tree.shared_adjacency_context
    if context is None:
        return "n/a"
    return enum_name(context.adjacency.shape)


tree_summary = pd.DataFrame(
    [
        {
            "construction": name,
            "live nodes": tree.num_nodes,
            "non-root nodes": tree.num_nodes - 1,
            "root altitude": tree.node_altitude(tree.root),
            "altitude order": enum_name(tree.node_altitude_order),
            "adjacency": adjacency_name(tree),
            "exact API reconstruction": np.array_equal(
                tree.reconstruct_from_node_altitudes(), image
            ),
        }
        for name, tree in trees.items()
    ]
).set_index("construction")

assert tree_summary["exact API reconstruction"].all()
display(tree_summary)

,live nodes,non-root nodes,root altitude,altitude order,adjacency,exact API reconstruction
construction,,,,,,
Unrestricted,6,5,1,UNCONSTRAINED,EUCLIDEAN_DISK,True
Saturated,6,5,0,UNCONSTRAINED,n/a,True


### 3. Read the recorded residual events

For every non-root node, the API exposes its connected support, valuation $\eta(X)$, parent, proper part, and signed residue

$$
r_k=\eta(X_k)-\eta(\triangleleft(X_k)).
$$

A positive residue is a regional-maximum event; a negative residue is a regional-minimum event.

In [4]:
def support_pixels(tree, node_id) -> np.ndarray:
    return np.fromiter(
        tree.node_support(node_id),
        dtype=np.int64,
    )


def residual_schedule(tree) -> list[dict]:
    root = tree.root
    events = []
    for node_id in tree.alive_node_ids:
        if node_id == root:
            continue
        pixels = support_pixels(tree, node_id)
        parent = tree.parent(node_id)
        residue = int(tree.node_residue(node_id))
        events.append(
            {
                "node": int(node_id),
                "support": pixels,
                "area": int(pixels.size),
                "spatial_min": int(pixels.min()),
                "eta": int(tree.node_altitude(node_id)),
                "parent": int(parent),
                "parent_eta": int(tree.node_altitude(parent)),
                "residue": residue,
                "polarity": "maximum" if residue > 0 else "minimum",
                "proper_part_area": int(tree.proper_part_cardinality(node_id)),
            }
        )

    events.sort(key=lambda event: (event["area"], event["spatial_min"]))
    for step, event in enumerate(events):
        event["k"] = step
    return events


event_schedules = {
    name: residual_schedule(tree) for name, tree in trees.items()
}


def event_table(events) -> pd.DataFrame:
    rows = []
    for event in events:
        minimum_row, minimum_column = divmod(event["spatial_min"], COLUMNS)
        rows.append(
            {
                "k": event["k"],
                "node": event["node"],
                "|X_k|": event["area"],
                "min X_k (row, column)": (minimum_row, minimum_column),
                "eta(X_k)": event["eta"],
                "eta(parent)": event["parent_eta"],
                "r_k": event["residue"],
                "polarity": event["polarity"],
                "|rho(X_k)|": event["proper_part_area"],
            }
        )
    return pd.DataFrame(rows).set_index("k")


for construction, events in event_schedules.items():
    print(construction)
    display(event_table(events))

Unrestricted


,node,|X_k|,"min X_k (row, column)",eta(X_k),eta(parent),r_k,polarity,|rho(X_k)|
k,,,,,,,,
0,1,19,"(3, 13)",0,1,-1,minimum,19
1,2,27,"(0, 16)",2,1,1,maximum,27
2,3,51,"(2, 7)",2,1,1,maximum,51
3,4,53,"(4, 6)",0,1,-1,minimum,53
4,5,120,"(0, 0)",0,1,-1,minimum,120


Saturated


,node,|X_k|,"min X_k (row, column)",eta(X_k),eta(parent),r_k,polarity,|rho(X_k)|
k,,,,,,,,
0,1,19,"(3, 13)",0,1,-1,minimum,19
1,2,27,"(0, 16)",2,1,1,maximum,27
2,3,53,"(4, 6)",0,2,-2,minimum,53
3,4,104,"(2, 7)",2,1,1,maximum,51
4,5,169,"(0, 16)",1,0,1,maximum,19


The first two events are common to both constructions: $Q$ rises from $0$ to $1$, then $D$ falls from $2$ to $1$. The schedules diverge at $k=2$. The unrestricted construction may lower the unsaturated ring, whereas the saturated construction first fills its hole.

### 4. Recover the two scale-spaces from Equation (5)

Starting at $I^0$, each recorded event applies

$$
I^{k+1}=I^k-r_k\mathbf{1}_{X_k}.
$$

This recovery uses only the valued residual tree: no component tree is rebuilt and no regional extremum is searched again.

In [5]:
def recover_scale_space(input_image, tree, events) -> list[np.ndarray]:
    states = [input_image.astype(np.int16)]
    for event in events:
        next_state = states[-1].copy()
        next_state.flat[event["support"]] -= event["residue"]
        states.append(next_state)

    root_altitude = int(tree.node_altitude(tree.root))
    assert np.all(states[-1] == root_altitude)
    return states


scale_spaces = {
    name: recover_scale_space(image, trees[name], events)
    for name, events in event_schedules.items()
}

fig, axes = plt.subplots(
    2,
    6,
    figsize=(15.5, 5.8),
    constrained_layout=True,
)

for row_index, construction in enumerate(("Unrestricted", "Saturated")):
    states = scale_spaces[construction]
    events = event_schedules[construction]
    for state_index, state in enumerate(states):
        axis = axes[row_index, state_index]
        axis.imshow(
            state,
            cmap=gray_cmap,
            norm=gray_norm,
            interpolation="nearest",
        )
        if state_index == 0:
            title = r"$I^0$"
        else:
            event = events[state_index - 1]
            title = (
                rf"$I^{{{state_index}}}$"
                + "\n"
                + rf"$r_{{{state_index - 1}}}={event['residue']}$, "
                + rf"$|X|={event['area']}$"
            )
        axis.set_title(title, fontsize=10)
        axis.set_axis_off()
    axes[row_index, 0].set_ylabel(construction, fontsize=12)

fig.suptitle("Scale-spaces recovered from the valued residual trees", fontsize=15)
plt.show()

/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42385/66337998.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The unrestricted sequence ends at the constant image $I^5\equiv1$. The saturated sequence ends at $I^5\equiv0$. Different final constants are compatible with exact reconstruction because the signed residual families differ.

### 5. Inspect the Hasse diagrams

The next plot shows the inclusion parent map of each recorded support. Labels report the node valuation, signed residue, and support area. Orange nodes carry positive residues (maximum events), blue nodes carry negative residues (minimum events), and the root is gray.

In [6]:
def support_sort_key(tree, node_id):
    pixels = support_pixels(tree, node_id)
    return (int(pixels.size), int(pixels.min()))


def tree_layout(tree):
    root = tree.root
    children = {
        node: sorted(tree.children(node), key=lambda child: support_sort_key(tree, child))
        for node in tree.alive_node_ids
    }
    positions = {}
    next_leaf_x = 0

    def place(node, depth):
        nonlocal next_leaf_x
        node_children = children[node]
        if not node_children:
            x_position = next_leaf_x
            next_leaf_x += 1
        else:
            child_positions = [place(child, depth + 1) for child in node_children]
            x_position = float(np.mean(child_positions))
        positions[node] = (x_position, -depth)
        return x_position

    place(root, 0)
    return positions, children


def plot_tree(axis, tree, title):
    positions, children = tree_layout(tree)
    root = tree.root

    for parent, node_children in children.items():
        for child in node_children:
            x_parent, y_parent = positions[parent]
            x_child, y_child = positions[child]
            axis.plot(
                [x_parent, x_child],
                [y_parent, y_child],
                color="#666666",
                linewidth=1.5,
                zorder=1,
            )

    for node_id, (x_position, y_position) in positions.items():
        pixels = support_pixels(tree, node_id)
        if node_id == root:
            color = "#d0d0d0"
            label = (
                "root"
                + "\n"
                + rf"$\eta={tree.node_altitude(node_id)}$"
                + "\n"
                + rf"$|X|={pixels.size}$"
            )
        else:
            residue = int(tree.node_residue(node_id))
            color = "#e69f5a" if residue > 0 else "#66a6d9"
            label = (
                rf"$n={node_id}$"
                + "\n"
                + rf"$\eta={tree.node_altitude(node_id)},\ r={residue}$"
                + "\n"
                + rf"$|X|={pixels.size}$"
            )
        axis.scatter(
            [x_position],
            [y_position],
            s=2250,
            color=color,
            edgecolor="#333333",
            linewidth=1.2,
            zorder=2,
        )
        axis.text(
            x_position,
            y_position,
            label,
            ha="center",
            va="center",
            fontsize=8.5,
            zorder=3,
        )

    axis.set_title(title, fontsize=13)
    axis.margins(x=0.12, y=0.35)
    axis.set_axis_off()


fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.2), constrained_layout=True)
for axis, (construction, tree) in zip(axes, trees.items()):
    plot_tree(axis, tree, construction)
fig.suptitle("Residual-tree Hasse diagrams", fontsize=15)
plt.show()

/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42385/2502884794.py:96: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The unrestricted example is a star: all five event supports are disjoint. Saturated eligibility changes the evolution, so later supports contain earlier ones and the final hierarchy has nested branches. In both cases, the tree is induced by the evolution rather than by taking a static union of upper and lower components.

### 6. Why saturation alone is not a tree of shapes

Figure 1(c) starts from an upper component $X=P\cup R$. Its saturation is $A=P\cup R\cup H$. The lower component $B=R\cup Q$ is already saturated. Under the shared 4-adjacency, $A$ and $B$ cross: they intersect, but neither contains the other.

In [7]:
FOUR_NEIGHBORS = ((-1, 0), (1, 0), (0, -1), (0, 1))


def exterior_component(mask, infinity_pixel=0) -> np.ndarray:
    # Return the 4-connected complement component containing p_infinity.
    complement = ~mask
    exterior = np.zeros_like(mask, dtype=bool)
    start = divmod(infinity_pixel, mask.shape[1])
    if not complement[start]:
        return exterior

    queue = deque([start])
    exterior[start] = True
    while queue:
        current_row, current_column = queue.popleft()
        for row_delta, column_delta in FOUR_NEIGHBORS:
            neighbor_row = current_row + row_delta
            neighbor_column = current_column + column_delta
            if (
                0 <= neighbor_row < mask.shape[0]
                and 0 <= neighbor_column < mask.shape[1]
                and complement[neighbor_row, neighbor_column]
                and not exterior[neighbor_row, neighbor_column]
            ):
                exterior[neighbor_row, neighbor_column] = True
                queue.append((neighbor_row, neighbor_column))
    return exterior


def saturate(mask, infinity_pixel=0) -> np.ndarray:
    if mask.flat[infinity_pixel]:
        return np.ones_like(mask, dtype=bool)
    return ~exterior_component(mask, infinity_pixel)


X = zone_masks["P"] | zone_masks["R"]
A = saturate(X, INFINITY_PIXEL)
B = zone_masks["R"] | zone_masks["Q"]

assert np.array_equal(A, zone_masks["P"] | zone_masks["R"] | zone_masks["H"])
assert np.array_equal(saturate(B, INFINITY_PIXEL), B)

intersection = A & B
crossing_checks = {
    "A intersects B": bool(intersection.any()),
    "A is not contained in B": bool((A & ~B).any()),
    "B is not contained in A": bool((B & ~A).any()),
}
assert all(crossing_checks.values())

overlap_view = np.zeros_like(image, dtype=np.uint8)
overlap_view[A & ~B] = 1
overlap_view[B & ~A] = 2
overlap_view[intersection] = 3
overlap_cmap = ListedColormap(["white", "#e7ad7f", "#78abd2", "#8c5aa8"])
overlap_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], overlap_cmap.N)

fig, axes = plt.subplots(1, 4, figsize=(12.8, 3.3), constrained_layout=True)
for axis, mask, title in zip(
    axes[:3],
    (X, A, B),
    (r"$X=P\cup R$", r"$A=\mathrm{Sat}(X)$", r"$B=R\cup Q$"),
):
    axis.imshow(mask, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
    axis.set_title(title)
    axis.set_axis_off()

axes[3].imshow(overlap_view, cmap=overlap_cmap, norm=overlap_norm, interpolation="nearest")
axes[3].set_title(r"purple: $A\cap B$")
axes[3].set_axis_off()
fig.suptitle("Two saturated sets that cross under the shared adjacency", fontsize=14)
plt.show()

display(pd.Series(crossing_checks, name="observed"))

/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42385/3293475574.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


A intersects B             True
A is not contained in B    True
B is not contained in A    True
Name: observed, dtype: bool

This does not contradict the laminarity of the saturated residual tree. The tree records supports after successive eliminations have modified the remaining candidates; it does not statically combine every saturated upper and lower component of $I^0$. Theorem 3 requires a separate admissible topographic convention with complementary adjacencies.

## Checks

### 7. Exact decomposition, laminarity, and saturation

The following checks are finite computations on this example, not substitutes for the general proofs. They test:

- the telescoping reconstruction in Equation (6);
- nondecreasing event-support cardinalities;
- pairwise laminarity of the recorded supports;
- non-empty proper parts;
- the bound $q\leq|\mathcal Z(I^0)|-1$;
- saturation of every non-root node in the saturated construction.

In [8]:
def residual_sum_reconstruction(tree, events) -> np.ndarray:
    root_altitude = int(tree.node_altitude(tree.root))
    reconstruction = np.full(image.shape, root_altitude, dtype=np.int16)
    for event in events:
        reconstruction.flat[event["support"]] += event["residue"]
    return reconstruction


def recorded_supports(tree) -> list[frozenset[int]]:
    return [
        frozenset(int(pixel) for pixel in support_pixels(tree, node_id))
        for node_id in tree.alive_node_ids
    ]


def is_laminar(supports) -> bool:
    for left_index, left in enumerate(supports):
        for right in supports[left_index + 1 :]:
            if left & right and not (left <= right or right <= left):
                return False
    return True


def is_saturated_support(tree, node_id) -> bool:
    mask = tree.reconstruct_node(node_id).astype(bool)
    return (
        not mask.flat[INFINITY_PIXEL]
        and np.array_equal(saturate(mask, INFINITY_PIXEL), mask)
    )


check_rows = []
for construction, tree in trees.items():
    events = event_schedules[construction]
    event_areas = [event["area"] for event in events]
    non_root_nodes = [
        node_id
        for node_id in tree.alive_node_ids
        if node_id != tree.root
    ]
    check_rows.append(
        {
            "construction": construction,
            "Equation (6) exact": np.array_equal(
                residual_sum_reconstruction(tree, events), image
            ),
            "areas nondecreasing": event_areas == sorted(event_areas),
            "supports laminar": is_laminar(recorded_supports(tree)),
            "proper parts non-empty": all(
                tree.proper_part_cardinality(node_id) > 0
                for node_id in tree.alive_node_ids
            ),
            "q <= |Z(I0)| - 1": len(events) <= len(zone_masks) - 1,
            "non-root supports saturated": (
                all(is_saturated_support(tree, node_id) for node_id in non_root_nodes)
                if construction == "Saturated"
                else "not required"
            ),
        }
    )

structural_checks = pd.DataFrame(check_rows).set_index("construction")
boolean_columns = structural_checks.select_dtypes(include="bool").columns
assert structural_checks[boolean_columns].to_numpy().all()
assert bool(structural_checks.loc["Saturated", "non-root supports saturated"])
display(structural_checks)

,Equation (6) exact,areas nondecreasing,supports laminar,proper parts non-empty,q <= |Z(I0)| - 1,non-root supports saturated
construction,,,,,,
Unrestricted,True,True,True,True,True,not required
Saturated,True,True,True,True,True,True


### 8. Contrast inversion and self-duality

For this image, $V=[0,2]$ and $s=2$. Proposition 3 predicts that $\bar I=s-I$ produces the same supports and Hasse diagram, while every signed residual changes sign and its magnitude is preserved.

In [9]:
contrast_sum = int(image.min()) + int(image.max())
inverted_image = (contrast_sum - image.astype(np.int16)).astype(np.uint8)

inverted_trees = {
    "Unrestricted": factory.create_unrestricted_residual_tree(
        inverted_image,
        radius=ADJACENCY_RADIUS,
    ),
    "Saturated": factory.create_saturated_residual_tree(
        inverted_image,
        infinity_pixel=INFINITY_PIXEL,
        radius=ADJACENCY_RADIUS,
    ),
}


def canonical_tree_data(tree):
    support_by_node = {
        int(node_id): frozenset(
            int(pixel) for pixel in support_pixels(tree, node_id)
        )
        for node_id in tree.alive_node_ids
    }
    root = int(tree.root)
    parent_by_support = {
        support_by_node[node_id]: support_by_node[int(tree.parent(node_id))]
        for node_id in tree.alive_node_ids
        if node_id != root
    }
    residue_by_support = {
        support_by_node[node_id]: int(tree.node_residue(node_id))
        for node_id in tree.alive_node_ids
        if node_id != root
    }
    return {
        "supports": set(support_by_node.values()),
        "parents": parent_by_support,
        "residues": residue_by_support,
        "root_altitude": int(tree.node_altitude(root)),
    }


self_duality_rows = []
for construction in trees:
    original_data = canonical_tree_data(trees[construction])
    inverted_data = canonical_tree_data(inverted_trees[construction])
    residuals_reverse = all(
        inverted_data["residues"][support] == -residue
        for support, residue in original_data["residues"].items()
    )
    self_duality_rows.append(
        {
            "construction": construction,
            "supports identical": original_data["supports"] == inverted_data["supports"],
            "Hasse diagram identical": original_data["parents"] == inverted_data["parents"],
            "residual signs reversed": residuals_reverse,
            "root valuation transformed": (
                inverted_data["root_altitude"]
                == contrast_sum - original_data["root_altitude"]
            ),
            "inverted reconstruction exact": np.array_equal(
                inverted_trees[construction].reconstruct_from_node_altitudes(), inverted_image
            ),
        }
    )

self_duality_checks = pd.DataFrame(self_duality_rows).set_index("construction")
assert self_duality_checks.to_numpy().all()

fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.3), constrained_layout=True)
for axis, displayed_image, title in zip(
    axes,
    (image, inverted_image),
    (r"$I$", r"$\bar I=2-I$"),
):
    axis.imshow(displayed_image, cmap=gray_cmap, norm=gray_norm, interpolation="nearest")
    axis.set_title(title)
    axis.set_axis_off()
plt.show()

display(self_duality_checks)

/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42385/1272528865.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,supports identical,Hasse diagram identical,residual signs reversed,root valuation transformed,inverted reconstruction exact
construction,,,,,
Unrestricted,True,True,True,True,True
Saturated,True,True,True,True,True


### 9. Prune the Figure 1 hierarchies by support area

The paper contracts every non-root node with $A(X)\leq\lambda$. The public `filtering_by_pruning_min` call keeps nodes whose area is at least its threshold, so for integer areas we pass `threshold = lambda + 1`.

The same $\lambda$ does not imply the same output across hierarchy families: each tree organizes a different support family.

In [10]:
filter_trees = {
    "Tree of Shapes": factory.create_tree_of_shapes(
        image,
        convention=mmcfilters.TopographicConvention(
            mmcfilters.ComplementaryGridImmersion(mmcfilters.ComplementaryAdjacencies(
                mmcfilters.RegularGridAdjacency2D(*image.shape, 1.0),
                mmcfilters.RegularGridAdjacency2D(*image.shape, 1.5),
            ))
        ),
    ),
    **trees,
}
AREA = mmcfilters.Attribute.AREA
LAMBDA_VALUES = (19, 53, 104)

filtered_images = {}
filtering_rows = []
for construction, tree in filter_trees.items():
    area = mmcfilters.Attribute.compute_single_topology_attribute(
        tree,
        AREA,
        dtype=np.float64,
    )
    filters = mmcfilters.AttributeFilters(tree)
    for lambda_value in LAMBDA_VALUES:
        filtered = filters.filtering_by_pruning_min(area, lambda_value + 1)
        filtered_images[(construction, lambda_value)] = filtered
        filtering_rows.append(
            {
                "construction": construction,
                "lambda": lambda_value,
                "changed pixels": int(np.count_nonzero(filtered != image)),
                "remaining gray levels": int(np.unique(filtered).size),
            }
        )

fig, axes = plt.subplots(
    len(LAMBDA_VALUES),
    1 + len(filter_trees),
    figsize=(12.5, 9.2),
    constrained_layout=True,
)
for row_index, lambda_value in enumerate(LAMBDA_VALUES):
    axes[row_index, 0].imshow(
        image,
        cmap=gray_cmap,
        norm=gray_norm,
        interpolation="nearest",
    )
    axes[row_index, 0].set_title("Input")
    axes[row_index, 0].set_ylabel(rf"$\lambda={lambda_value}$", fontsize=12)
    axes[row_index, 0].set_axis_off()

    for column_index, construction in enumerate(filter_trees, start=1):
        axes[row_index, column_index].imshow(
            filtered_images[(construction, lambda_value)],
            cmap=gray_cmap,
            norm=gray_norm,
            interpolation="nearest",
        )
        axes[row_index, column_index].set_title(construction)
        axes[row_index, column_index].set_axis_off()

fig.suptitle(r"Area pruning: contract nodes with $|X|\leq\lambda$", fontsize=15)
plt.show()

filtering_summary = pd.DataFrame(filtering_rows).set_index(
    ["construction", "lambda"]
)
display(filtering_summary)

/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42385/1004737723.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


changed pixels  remaining gray levels
construction   lambda                                       
Tree of Shapes 19                 116                      3
               53                 169                      2
               104                169                      2
Unrestricted   19                  19                      3
               53                 150                      2
               104                150                      2
Saturated      19                  19                      3
               53                  99                      3
               104                150                      2

### 10. Inspect adjacency effects on the Figure 1 example

Figure 2 emphasizes that the shared adjacency is part of the model. The table below rebuilds both residual trees with shared 4-adjacency, shared 8-adjacency, and the centered rectangular $3\times11$ adjacency used in the paper. Larger neighborhoods merge some flat zones and therefore change the recorded supports.

In [11]:
rectangular_3x11 = mmcfilters.RegularGridAdjacency2D.rectangular(
    ROWS,
    COLUMNS,
    1,
    5,
)

adjacency_specs = {
    "shared 4": 1.0,
    "shared 8": 1.5,
    "shared 3x11": rectangular_3x11,
}


def build_residual_tree(mode, adjacency):
    if isinstance(adjacency, float):
        if mode == "Unrestricted":
            return factory.create_unrestricted_residual_tree(image, radius=adjacency)
        return factory.create_saturated_residual_tree(
            image,
            infinity_pixel=INFINITY_PIXEL,
            radius=adjacency,
        )
    if mode == "Unrestricted":
        return factory.create_unrestricted_residual_tree(image, adjacency)
    return factory.create_saturated_residual_tree(
        image,
        adjacency,
        INFINITY_PIXEL,
    )


adjacency_rows = []
for adjacency_name, adjacency in adjacency_specs.items():
    for mode in ("Unrestricted", "Saturated"):
        tree = build_residual_tree(mode, adjacency)
        support_areas = [
            int(support_pixels(tree, node_id).size)
            for node_id in tree.alive_node_ids
            if node_id != tree.root
        ]
        adjacency_rows.append(
            {
                "adjacency": adjacency_name,
                "construction": mode,
                "live nodes": tree.num_nodes,
                "root altitude": int(tree.node_altitude(tree.root)),
                "non-root support areas": sorted(support_areas),
                "exact reconstruction": np.array_equal(
                    tree.reconstruct_from_node_altitudes(), image
                ),
            }
        )

adjacency_summary = pd.DataFrame(adjacency_rows).set_index(
    ["adjacency", "construction"]
)
assert adjacency_summary["exact reconstruction"].all()
display(adjacency_summary)

live nodes  root altitude  non-root support areas  \
adjacency   construction                                                      
shared 4    Unrestricted           6              1   [19, 27, 51, 53, 120]   
            Saturated              6              0  [19, 27, 53, 104, 169]   
shared 8    Unrestricted           4              1          [53, 131, 139]   
            Saturated              4              0          [53, 131, 150]   
shared 3x11 Unrestricted           3              0                [78, 97]   
            Saturated              3              0                [78, 97]   

                          exact reconstruction  
adjacency   construction                        
shared 4    Unrestricted                  True  
            Saturated                     True  
shared 8    Unrestricted                  True  
            Saturated                     True  
shared 3x11 Unrestricted                  True  
            Saturated                     True

## Figure 2: hydrant area-filtering experiment

The following cells reproduce the experimental configuration of Figure 2 on
the paper's original $363\times352$-pixel grayscale hydrant image. The five
columns are:

1. the tree of shapes with complementary Min4/Max8 connectivity;
2. the unrestricted residual tree with shared 8-adjacency;
3. the saturated residual tree with shared 8-adjacency;
4. the unrestricted residual tree with a centered shared $3\times11$ adjacency;
5. the saturated residual tree with that shared $3\times11$ adjacency.

Residual construction uses row-major order, the default contrast-invariant tie
policy, and the upper-left pixel as $p_\infty$ for the saturated variants. The
three paper thresholds are $\lambda/|P|\in\{0.5\%,12\%,15\%\}$. As above, the
API receives $\lambda+1$ because the paper contracts nodes with
$A(X)\leq\lambda$.

### 11. Load the original experiment image and parameters

In [12]:
def first_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Run the notebook from the repository root or the notebooks directory."
    )


hydrant_path = first_existing_path(
    Path("../dat/hydrant.png"),
    Path("dat/hydrant.png"),
)
hydrant_image = cv2.imread(str(hydrant_path), cv2.IMREAD_GRAYSCALE)
assert hydrant_image is not None
hydrant_image = np.ascontiguousarray(hydrant_image, dtype=np.uint8)

HYDRANT_ROWS, HYDRANT_COLUMNS = hydrant_image.shape
assert (HYDRANT_COLUMNS, HYDRANT_ROWS) == (363, 352)

HYDRANT_PERCENTAGES = (0.5, 12.0, 15.0)
HYDRANT_LAMBDAS = tuple(
    int(round(percentage / 100.0 * hydrant_image.size))
    for percentage in HYDRANT_PERCENTAGES
)
assert HYDRANT_LAMBDAS == (639, 15_333, 19_166)

experiment_parameters = pd.Series(
    {
        "image": hydrant_path.as_posix(),
        "width x height": f"{HYDRANT_COLUMNS} x {HYDRANT_ROWS}",
        "pixels |P|": hydrant_image.size,
        "lambda values": HYDRANT_LAMBDAS,
        "saturated exterior pixel": 0,
    },
    name="Figure 2",
)

fig, ax = plt.subplots(figsize=(4.8, 4.8), constrained_layout=True)
ax.imshow(hydrant_image, cmap="gray", vmin=0, vmax=255)
ax.set_title("Figure 2 input")
ax.set_axis_off()
plt.show()

display(experiment_parameters.to_frame())

/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42385/3927961328.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Figure 2
image,../dat/hydrant.png
width x height,363 x 352
pixels |P|,127776
lambda values,"(639, 15333, 19166)"
saturated exterior pixel,0


### 12. Construct the five hierarchies

In [13]:
hydrant_rectangular_3x11 = mmcfilters.RegularGridAdjacency2D.rectangular(
    HYDRANT_ROWS,
    HYDRANT_COLUMNS,
    1,
    5,
)

hydrant_trees = {
    "Tree of Shapes\nMin4/Max8": factory.create_tree_of_shapes(
        hydrant_image,
        convention=mmcfilters.TopographicConvention(
            mmcfilters.ComplementaryGridImmersion(mmcfilters.ComplementaryAdjacencies(
                mmcfilters.RegularGridAdjacency2D(*hydrant_image.shape, 1.0),
                mmcfilters.RegularGridAdjacency2D(*hydrant_image.shape, 1.5),
            ))
        ),
    ),
    "Unrestricted\nshared 8": factory.create_unrestricted_residual_tree(
        hydrant_image,
        radius=1.5,
    ),
    "Saturated\nshared 8": factory.create_saturated_residual_tree(
        hydrant_image,
        infinity_pixel=0,
        radius=1.5,
    ),
    "Unrestricted\nshared 3x11": factory.create_unrestricted_residual_tree(
        hydrant_image,
        hydrant_rectangular_3x11,
    ),
    "Saturated\nshared 3x11": factory.create_saturated_residual_tree(
        hydrant_image,
        hydrant_rectangular_3x11,
        0,
    ),
}

# Every hierarchy here keeps the input scale: the complementary-grid tree of
# shapes publishes unchanged 8-bit source levels, as the residual trees do.
def reconstructs_input_losslessly(tree) -> bool:
    reconstruction = tree.reconstruct_from_node_altitudes()
    return np.array_equal(reconstruction, hydrant_image.astype(reconstruction.dtype))


hydrant_tree_summary = pd.DataFrame(
    [
        {
            "hierarchy": name.replace("\n", " "),
            "live nodes": tree.num_nodes,
            "root altitude": int(tree.node_altitude(tree.root)),
            "exact reconstruction": reconstructs_input_losslessly(tree),
        }
        for name, tree in hydrant_trees.items()
    ]
).set_index("hierarchy")

assert hydrant_tree_summary["exact reconstruction"].all()
display(hydrant_tree_summary)

,live nodes,root altitude,exact reconstruction
hierarchy,,,
Tree of Shapes Min4/Max8,22137,268,True
Unrestricted shared 8,17089,74,True
Saturated shared 8,17104,134,True
Unrestricted shared 3x11,5809,75,True
Saturated shared 3x11,5809,134,True


### 13. Apply the three area thresholds

In [14]:
hydrant_filtered_images = {}
hydrant_filtering_rows = []

for hierarchy_name, tree in hydrant_trees.items():
    area = mmcfilters.Attribute.compute_single_topology_attribute(
        tree,
        AREA,
        dtype=np.float64,
    )
    filters = mmcfilters.AttributeFilters(tree)
    for percentage, lambda_value in zip(
        HYDRANT_PERCENTAGES,
        HYDRANT_LAMBDAS,
    ):
        filtered = filters.filtering_by_pruning_min(area, lambda_value + 1)
        hydrant_filtered_images[(hierarchy_name, lambda_value)] = filtered
        hydrant_filtering_rows.append(
            {
                "hierarchy": hierarchy_name.replace("\n", " "),
                "lambda / |P|": f"{percentage:g}%",
                "lambda": lambda_value,
                "changed pixels": int(
                    np.count_nonzero(filtered != hydrant_image)
                ),
                "remaining gray levels": int(np.unique(filtered).size),
            }
        )

fig, axes = plt.subplots(
    len(HYDRANT_LAMBDAS),
    1 + len(hydrant_trees),
    figsize=(17.0, 9.2),
    constrained_layout=True,
)

for row_index, (percentage, lambda_value) in enumerate(
    zip(HYDRANT_PERCENTAGES, HYDRANT_LAMBDAS)
):
    input_axis = axes[row_index, 0]
    if row_index == 1:
        input_axis.imshow(hydrant_image, cmap="gray", vmin=0, vmax=255)
        input_axis.set_title("Input")
    input_axis.set_axis_off()

    for column_index, hierarchy_name in enumerate(hydrant_trees, start=1):
        axis = axes[row_index, column_index]
        axis.imshow(
            hydrant_filtered_images[(hierarchy_name, lambda_value)],
            cmap="gray",
            vmin=0,
            vmax=255,
        )
        if row_index == 0:
            axis.set_title(hierarchy_name, fontsize=10)
        if column_index == 1:
            axis.set_ylabel(
                rf"${percentage:g}\%$" + "\n" + rf"$\lambda={lambda_value:,}$",
                fontsize=10,
            )
        axis.set_xticks([])
        axis.set_yticks([])

fig.suptitle(
    r"Figure 2 configuration: area pruning with $A(X)\leq\lambda$",
    fontsize=15,
)
plt.show()

hydrant_filtering_summary = pd.DataFrame(hydrant_filtering_rows).set_index(
    ["hierarchy", "lambda / |P|"]
)
display(hydrant_filtering_summary)

/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42385/1101181032.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


lambda  changed pixels  \
hierarchy                lambda / |P|                           
Tree of Shapes Min4/Max8 0.5%             639          127692   
                         12%            15333          127745   
                         15%            19166          127743   
Unrestricted shared 8    0.5%             639           42020   
                         12%            15333          102383   
                         15%            19166          112403   
Saturated shared 8       0.5%             639           42014   
                         12%            15333          101753   
                         15%            19166          108514   
Unrestricted shared 3x11 0.5%             639           22444   
                         12%            15333           83371   
                         15%            19166           95197   
Saturated shared 3x11    0.5%             639           22437   
                         12%            15333           83364   
                         15%            19166           92062   

                                       remaining gray levels  
hierarchy                lambda / |P|                         
Tree of Shapes Min4/Max8 0.5%                            133  
                         12%                              82  
                         15%                              70  
Unrestricted shared 8    0.5%                            133  
                         12%                              82  
                         15%                              69  
Saturated shared 8       0.5%                            133  
                         12%                              82  
                         15%                              70  
Unrestricted shared 3x11 0.5%                            132  
                         12%                              82  
                         15%                              76  
Saturated shared 3x11    0.5%                            132  
                         12%                              82  
                         15%                              76

The reproduced grid makes the adjacency effect directly visible. With shared 8-adjacency, the unrestricted hierarchy retains cavities that saturation fills relative to the frame. The centered $3\times11$ neighborhood can connect regions that are distinct under 8-adjacency, so its unrestricted and saturated results become more similar at the larger thresholds.

## Next steps

- Replace the synthetic image while preserving `np.uint8`, two-dimensional shape, and C-contiguity.
- Change `ADJACENCY_RADIUS`, `INFINITY_PIXEL`, or the rectangular structuring element and rerun all checks.
- Inspect which event first changes when two candidates have equal area.
- Compare additional increasing node attributes with area pruning.
- Use the paper for the general arguments: this notebook verifies one finite instance and intentionally does not reproduce the proofs.

The central implementation bridge is now explicit: a non-root tree node stores $X_k$, `nodeAltitude(node)` stores $\eta(X_k)$, `nodeResidue(node)` stores $r_k$, and `reconstructNode(node)` exposes the support mask used in Equations (5) and (6).